In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PowerTransformer
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib

In [ ]:
# Load dataset
df = pd.read_csv("data.csv")

print("Columns:", df.columns)
print("Missing values:\n", df.isnull().sum())

X = df[['HD', 'DBT', 'RH', 'DSR', 'DiSR', 'GHR', 'LA']].values
y = df[['Energy']].values

# Check the range of inputs and outputs
print("\nInput features' ranges:")
print("HD (min, max):", X[:, 0].min(), X[:, 0].max())
print("DBT (min, max):", X[:, 1].min(), X[:, 1].max())
print("RH (min, max):", X[:, 2].min(), X[:, 2].max())
print("DiSR (min, max):", X[:, 3].min(), X[:, 3].max())
print("GHR (min, max):", X[:, 4].min(), X[:, 4].max())
print("LA (min, max):", X[:, 5].min(), X[:, 5].max())
print("\nTargets' ranges:")
print("Energy (min, max):", y.min(), y.max())

In [ ]:
# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# Normalize features and targets
scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)
joblib.dump(scaler_X, 'scaler_input_energy.pkl')


# Check the scale of features and targets
print("\nInput features' ranges:")
print("HD (min, max):", X_train_scaled[:, 0].min(), X_train_scaled[:, 0].max())
print("DBT (min, max):", X_train_scaled[:, 1].min(), X_train_scaled[:, 1].max())
print("RH (min, max):", X_train_scaled[:, 2].min(), X_train_scaled[:, 2].max())
print("DiSR (min, max):", X_train_scaled[:, 3].min(), X_train_scaled[:, 3].max())
print("GHR (min, max):", X_train_scaled[:, 4].min(), X_train_scaled[:, 4].max())
print("LA (min, max):", X_train_scaled[:, 5].min(), X_train_scaled[:, 5].max())
print("\nTargets' ranges:")
print("Energy (min, max):", y.min(), y.max())

In [ ]:
# Build model
model = keras.Sequential([
    layers.Dense(512, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    layers.BatchNormalization(),
    layers.Dropout(0.05),

    layers.Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),

    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.BatchNormalization(),

    layers.Dense(1, activation='linear')
])

optimizer = keras.optimizers.RMSprop(
    learning_rate=keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=0.005,
        decay_steps=100,
        decay_rate=0.9
    )
)

model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

# Training
early_stopping = keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True)

history = model.fit(
    X_train_scaled, y_train,
    epochs=300,
    batch_size=128,
    validation_data=(X_test_scaled, y_test),
    callbacks=[early_stopping],
    verbose=1
)

joblib.dump(model, 'Energy_Model.pkl')

In [ ]:
# Predict and inverse scaling
y_pred = model.predict(X_test_scaled)


y_pred_train = model.predict(X_train_scaled)


# Compare predictions vs actual (first 5 samples)
print("\nSample Predictions vs Actual:")
for i in range(5):
    print(f"Sample {i+1}: Predicted Energy={y_pred[i, 0]:.2f}, Actual Energy={y_test[i, 0]:.2f}")

# Metrics
train_loss = history.history['loss']      
train_mae = history.history['mae']         
val_loss = history.history['val_loss']    
val_mae = history.history['val_mae']   

# Print final epoch metrics
print(f"\nFinal Training Loss (MSE): {train_loss[-1]:.4f}")
print(f"Final Training MAE: {train_mae[-1]:.4f}")
print(f"Final Validation Loss (MSE): {val_loss[-1]:.4f}")
print(f"Final Validation MAE: {val_mae[-1]:.4f}")

# Original scale metrics
r2_model_train = r2_score(y_train, y_pred_train)

r2_model = r2_score(y_test, y_pred)

mae_model= mean_absolute_error(y_test, y_pred)

mse_model = mean_squared_error(y_test, y_pred)


print(f"\nR² for model_train: {r2_model_train:.3f}")

print(f"R² for model: {r2_model:.3f}")

print(f"\nMAE for model: {mae_model:.3f}")

print(f"\nMSE for model: {mse_model:.3f}")

In [ ]:
import pandas as pd

y_pred = model.predict(X_test_scaled)


# Create DataFrames
df_test = pd.DataFrame({
    'Actual_Energy': y_test[:, 0],
    'Predicted_Energy': y_pred[:, 0]
})


# Save to CSV
df_test.to_csv('test_predictions.csv', index=False)

161/161 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [ ]:
# Plot training curves
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('MSE Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['mae'], label='Training MAE')
plt.plot(history.history['val_mae'], label='Validation MAE')
plt.xlabel('Epochs')
plt.ylabel('MAE')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot of predictions vs true values
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.scatter(y_test[:, 0], y_pred[:, 0], alpha=0.5)
plt.xlabel('True Energy')
plt.ylabel('Predicted Energy')
plt.title('Energy Prediction')

In [ ]:
# Check target distributions
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.hist(y_train, bins=30)  # Ev distribution
plt.title('Energy Distribution')